# Lesson 2: Data Engineering — Homework

В реальному світі документи бувають зламані, з неправильним кодуванням, порожні, з підміненими розширеннями. Ваша задача — навчитись їх обробляти.

**6 завдань.** Кожне має `# TODO` — дописуйте реалізацію.

```bash
pip install -r requirements.txt
```

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
cd /content/drive/MyDrive/homework/

/content/drive/MyDrive/homework


In [3]:
ls

evaluate.py  homework.ipynb  README.md  requirements.txt  samples/


In [4]:
!apt-get update
!apt-get install -y poppler-utils libmagic-dev
%pip install -U pip setuptools wheel
%pip install "unstructured[docx,pdf,xlsx]"
%pip install langchain-text-splitters charset-normalizer filetype pdfplumber tenacity watchdog PyYAML openpyxl python-docx pytest nbformat packaging ipykernel

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,917 B in 2s (1,730 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libmagic-dev is already the newest version (1:5.41

In [5]:
from pathlib import Path

folder = Path("/content/drive/MyDrive/homework/samples/enterprise_challenges")

print("Файли для роботи:")
for f in sorted(folder.iterdir()):
    size = f.stat().st_size
    print(f"  {f.name:<40} {size:>10,} bytes")

Файли для роботи:
  actually_html.pdf                                99 bytes
  actually_pdf.html                                66 bytes
  binary_garbage.pdf                            1,024 bytes
  boilerplate_heavy.html                        5,344 bytes
  broken_archive.docx                           4,400 bytes
  complex_merged_formulas.xlsx                  9,275 bytes
  corrupted.xlsx                                5,100 bytes
  corrupted_truncated.pdf                         335 bytes
  empty_file.docx                                   0 bytes
  empty_file.html                                   0 bytes
  empty_file.pdf                                    0 bytes
  empty_with_headers.xlsx                       4,862 bytes
  financial_report_table.pdf                    2,783 bytes
  huge_audit_log.txt                        1,350,998 bytes
  huge_export.xlsx                            410,512 bytes
  huge_repetitive.html                        994,473 bytes
  latin1_mixed.html   

---
## Завдання 1: Визначення кодування файлів

Legacy-системи часто віддають файли в Windows-1251, Latin-1, UTF-8 з BOM — без вказання charset.
Якщо відкрити такий файл з неправильним кодуванням, отримаємо кракозябри (mojibake).

**Файли для роботи:**
- `utf8_with_bom.html` — UTF-8 з BOM маркером (зайві байти `\xef\xbb\xbf` на початку)
- `windows1251_no_charset.html` — Windows-1251 без мета-тегу charset (українські legacy-системи)
- `latin1_mixed.html` — Latin-1 з німецькими/французькими символами

**Що потрібно зробити:**
1. Прочитати файл як bytes
2. Визначити кодування через `charset_normalizer`
3. Декодувати текст правильно
4. Якщо є BOM — прибрати його

In [6]:
from charset_normalizer import from_bytes

def detect_and_read(file_path: str) -> dict:
    """
    Читає файл, визначає кодування, повертає декодований текст.

    Повертає:
    {
        "file": ім'я файлу,
        "encoding": визначене кодування,
        "had_bom": True/False,
        "text": декодований текст,
        "char_count": кількість символів
    }
    """
    file_path = Path(file_path)
    raw = file_path.read_bytes()

    # TODO: перевірити чи файл починається з BOM (b"\xef\xbb\xbf")
    # Якщо так — прибрати BOM з raw bytes
    bom = b"\xef\xbb\xbf"
    had_bom = raw.startswith(bom)
    if had_bom:
        raw = raw[len(bom):]

    # TODO: використати from_bytes(raw).best() для визначення кодування
    # result.encoding — назва кодування, str(result) — декодований текст
    # Якщо best() повернув None — декодувати як utf-8 з errors="replace"
    result = from_bytes(raw).best()
    if result is None:
        encoding = "utf-8"
        text = raw.decode("utf-8", errors="replace")
    else:
        encoding = result.encoding
        text = str(result)

    return {
        "file": file_path.name,
        "encoding": encoding,
        "had_bom": had_bom,
        "text": text,
        "char_count": len(text),
    }

In [7]:
# Тест
encoding_files = [
    "samples/enterprise_challenges/utf8_with_bom.html",
    "samples/enterprise_challenges/windows1251_no_charset.html",
    "samples/enterprise_challenges/latin1_mixed.html",
]

for f in encoding_files:
    result = detect_and_read(f)
    print(f"{result['file']}:")
    print(f"  Кодування: {result['encoding']}")
    print(f"  BOM: {result['had_bom']}")
    print(f"  Перші 150 символів: {result['text'][:150]}")
    print()

utf8_with_bom.html:
  Кодування: utf_8
  BOM: True
  Перші 150 символів: <html><head><meta charset="utf-8"><title>BOM Test</title></head><body><p>This file has a UTF-8 BOM marker.</p><p>Деякі символи українською мовою.</p><

windows1251_no_charset.html:
  Кодування: cp1251
  BOM: False
  Перші 150 символів: <html><head><title>Звіт</title></head><body><p>Цей документ в кодуванні Windows-1251 без мета-тегу charset.</p><p>Типова проблема з legacy системами.<

latin1_mixed.html:
  Кодування: cp1250
  BOM: False
  Перші 150 символів: <html><head><title>Rapport</title></head><body><p>Geräteübersicht und Maßnahmen.</p><p>Résumé des résultats français.</p></body></html>



---
## Завдання 2: Визначення реального типу файлу (magic bytes)

Розширення файлу може брехати. Хтось зберіг HTML як `.pdf`, або binary dump перейменував в `.pdf`.
Єдиний надійний спосіб — перевірити **magic bytes** (перші байти файлу, які визначають формат).

**Файли для роботи:**
- `actually_html.pdf` — HTML контент зі розширенням .pdf
- `actually_pdf.html` — PDF контент зі розширенням .html
- `binary_garbage.pdf` — рандомні байти з розширенням .pdf
- `empty_file.pdf` — порожній файл (0 байт)

**Що потрібно зробити:**
1. Визначити тип файлу по розширенню (declared type)
2. Визначити реальний тип через бібліотеку `filetype` (magic bytes)
3. Для HTML файлів (без magic bytes) — перевірити чи контент починається з `<html` або `<!doctype`
4. Повернути чи є mismatch

In [8]:
import filetype as filetype_lib

def detect_file_type(file_path: str) -> dict:
    """
    Визначає реальний тип файлу і порівнює з розширенням.

    Повертає:
    {
        "file": ім'я файлу,
        "declared_type": тип по розширенню,
        "detected_type": реальний тип (magic bytes),
        "is_mismatch": True якщо типи не збігаються,
        "issue": опис проблеми або None
    }
    """
    file_path = Path(file_path)
    declared_type = file_path.suffix.lstrip(".").lower()

    # TODO: перевірити чи файл порожній — якщо так, повернути issue "empty file"
    if file_path.stat().st_size == 0:
        return {
            "file": file_path.name,
            "declared_type": declared_type,
            "detected_type": None,
            "is_mismatch": True,
            "issue": "empty file",
        }

    # TODO: використати filetype_lib.guess(str(file_path)) для визначення типу
    # guess() повертає об'єкт з .extension або None якщо не вдалось визначити
    kind = filetype_lib.guess(str(file_path))
    detected_type = kind.extension if kind else None

    # TODO: якщо filetype не визначив (None) — перевірити вручну чи це HTML
    # Прочитати перші 512 байт і перевірити чи є b"<html" або b"<!doctype"
    if detected_type is None:
        head = file_path.read_bytes()[:512].lower().lstrip()
        if head.startswith(b"<html") or head.startswith(b"<!doctype"):
            detected_type = "html"

    # TODO: визначити чи є mismatch між declared_type і detected_type
    is_mismatch = detected_type is None or declared_type != detected_type
    issue = "type mismatch" if is_mismatch else None

    return {
        "file": file_path.name,
        "declared_type": declared_type,
        "detected_type": detected_type,
        "is_mismatch": is_mismatch,
        "issue": issue,
    }

In [9]:
# Тест
test_files = [
    "samples/enterprise_challenges/actually_html.pdf",
    "samples/enterprise_challenges/actually_pdf.html",
    "samples/enterprise_challenges/binary_garbage.pdf",
    "samples/enterprise_challenges/empty_file.pdf",
    "samples/enterprise_challenges/normal_report.xlsx",
]

for f in test_files:
    if Path(f).exists():
        result = detect_file_type(f)
        marker = "MISMATCH" if result["is_mismatch"] else "OK"
        print(f"  [{marker}] {result['file']}: declared={result['declared_type']}, real={result['detected_type']}")
        if result["issue"]:
            print(f"          -> {result['issue']}")

  [MISMATCH] actually_html.pdf: declared=pdf, real=html
          -> type mismatch
  [MISMATCH] actually_pdf.html: declared=html, real=pdf
          -> type mismatch
  [MISMATCH] binary_garbage.pdf: declared=pdf, real=None
          -> type mismatch
  [MISMATCH] empty_file.pdf: declared=pdf, real=None
          -> empty file
  [OK] normal_report.xlsx: declared=xlsx, real=xlsx


---
## Завдання 3: Витягування тексту з брудного HTML

Enterprise HTML — це жах: 50 рівнів вкладеності від Word-експорту, navigation bars, sidebars, scripts, cookie banners. Корисного тексту — 5%.

**Файли для роботи:**
- `malformed_deeply_nested.html` — 50 рівнів `<div>`, незакриті теги, зламані entities
- `boilerplate_heavy.html` — 30 nav items, 20 sidebar widgets, 3 абзаци корисного тексту

**Що потрібно зробити:**
1. Прочитати HTML через BeautifulSoup
2. Видалити `<script>`, `<style>`, `<nav>`, `<header>`, `<footer>` теги
3. Витягти чистий текст
4. Порахувати "корисність" — відсоток тексту від розміру HTML

In [10]:
from bs4 import BeautifulSoup

def extract_clean_text(file_path: str) -> dict:
    """
    Витягує чистий текст з HTML, видаляючи шум.

    Повертає:
    {
        "file": ім'я файлу,
        "raw_size": розмір HTML в символах,
        "text": чистий текст,
        "text_size": розмір тексту в символах,
        "useful_ratio": відсоток корисного тексту (text_size / raw_size)
    }
    """
    file_path = Path(file_path)
    raw_html = file_path.read_text(errors="replace")

    # TODO: створити BeautifulSoup з парсером "html.parser"
    soup = BeautifulSoup(raw_html, "html.parser")

    # TODO: видалити всі теги які є шумом: script, style, nav, header, footer, aside
    # Підказка: soup.find_all("script") повертає список, для кожного — tag.decompose()
    noise_tags = ["script", "style", "nav", "header", "footer", "aside"]
    for tag_name in noise_tags:
        for tag in soup.find_all(tag_name):
            tag.decompose()

    # TODO: витягти текст через soup.get_text(separator="\n", strip=True)
    text = soup.get_text(separator="\n", strip=True)

    raw_size = len(raw_html)
    text_size = len(text)
    useful_ratio = text_size / raw_size if raw_size > 0 else 0

    return {
        "file": file_path.name,
        "raw_size": raw_size,
        "text": text,
        "text_size": text_size,
        "useful_ratio": useful_ratio,
    }

In [11]:
# Тест
html_files = [
    "samples/enterprise_challenges/malformed_deeply_nested.html",
    "samples/enterprise_challenges/boilerplate_heavy.html",
    "samples/enterprise_challenges/multilingual.html",
]

for f in html_files:
    result = extract_clean_text(f)
    print(f"{result['file']}:")
    print(f"  HTML: {result['raw_size']:,} символів -> Текст: {result['text_size']:,} символів")
    print(f"  Корисність: {result['useful_ratio']:.1%}")
    print(f"  Текст: {result['text'][:200]}...")
    print()

malformed_deeply_nested.html:
  HTML: 1,461 символів -> Текст: 282 символів
  Корисність: 19.3%
  Текст: Enterprise Report Q4
Important quarterly data buried under 50 levels of nesting.
Revenue increased by 15% year-over-year.
Metric
Value
Revenue
$1.2M
Users
45,000
Profit & Loss for Q4 — results are © c...

boilerplate_heavy.html:
  HTML: 5,342 символів -> Текст: 635 символів
  Корисність: 11.9%
  Текст: Company Intranet - Important Policy Update
Important Policy Update: Data Handling Procedures
Effective March 1, 2026, all departments must follow updated data handling
                procedures for p...

multilingual.html:
  HTML: 885 символів -> Текст: 675 символів
  Корисність: 76.3%
  Текст: Multilingual Report
Quarterly Report / Щоквартальний звіт / 四半期報告
English
Revenue grew by 12% in Q4 2025, driven primarily by expansion into
new markets in Southeast Asia and Eastern Europe.
Українськ...



---
## Завдання 4: Обробка зламаних файлів — safe parser

Файли ламаються: обрізані PDF, corrupted DOCX (зламаний ZIP), binary garbage з розширенням .pdf, порожні файли, захищені паролем PDF.

Парсер не повинен падати — він має повертати результат або зрозумілу помилку.

**Файли для роботи:** вся папка `enterprise_challenges/` — там є і робочі, і зламані файли.

**Що потрібно зробити:**
1. Перевірити файл (порожній? правильний тип?)
2. Спробувати спарсити через `unstructured.partition.auto.partition`
3. Якщо помилка — зловити exception і повернути опис проблеми
4. Класифікувати помилку: `empty`, `corrupted`, `type_mismatch`, `parse_error`

In [12]:
from unstructured.partition.auto import partition

def safe_parse(file_path: str) -> dict:
    """
    Безпечно парсить документ — ніколи не падає, завжди повертає результат.

    Повертає при успіху:
    {
        "file": ім'я, "status": "ok",
        "text": витягнутий текст, "char_count": кількість символів
    }

    Повертає при помилці:
    {
        "file": ім'я, "status": "error",
        "error_type": тип помилки (empty/corrupted/type_mismatch/parse_error),
        "error_message": опис
    }
    """
    file_path = Path(file_path)

    # TODO: перевірити чи файл порожній (0 байт)
    # Якщо так — повернути status="error", error_type="empty"
    if file_path.stat().st_size == 0:
        return {
            "file": file_path.name,
            "status": "error",
            "error_type": "empty",
            "error_message": "file is empty",
        }

    # TODO: перевірити тип файлу (можна використати detect_file_type() з завдання 2,
    #   або просто filetype_lib.guess() напряму)
    # Якщо mismatch — повернути status="error", error_type="type_mismatch"
    ft = detect_file_type(str(file_path))
    plain_text_ext = {"txt", "log", "md", "csv"}
    skip_mismatch = ft["declared_type"] in plain_text_ext and ft["detected_type"] is None
    if ft["is_mismatch"] and not skip_mismatch:
        detail = ft.get("issue") or (
            "declared ."
            + str(ft["declared_type"])
            + ", detected "
            + str(ft["detected_type"])
        )
        return {
            "file": file_path.name,
            "status": "error",
            "error_type": "type_mismatch",
            "error_message": detail,
        }

    # TODO: спробувати спарсити файл через partition(filename=str(file_path))
    # Обгорнути в try/except Exception
    # При успіху — з'єднати елементи в текст і повернути status="ok"
    # При помилці — повернути status="error", error_type="parse_error" або "corrupted"
    try:
        elements = partition(filename=str(file_path))
        text = "\n".join(str(el) for el in elements)
        return {
            "file": file_path.name,
            "status": "ok",
            "text": text,
            "char_count": len(text),
        }
    except Exception as e:
        msg = str(e)
        low = msg.lower()
        if any(
            k in low
            for k in ("corrupt", "zip", "invalid", "truncat", "damaged", "malformed")
        ):
            err_type = "corrupted"
        elif "password" in low or "encrypt" in low:
            err_type = "parse_error"
        else:
            err_type = "parse_error"
        return {
            "file": file_path.name,
            "status": "error",
            "error_type": err_type,
            "error_message": msg,
        }

In [13]:
# Тест: прогнати safe_parse по всій папці enterprise_challenges
results = []
challenges_dir = Path("samples/enterprise_challenges")

for f in sorted(challenges_dir.iterdir()):
    if f.is_file():
        result = safe_parse(str(f))
        results.append(result)

ok = [r for r in results if r["status"] == "ok"]
errors = [r for r in results if r["status"] == "error"]

print(f"=== Результати: {len(ok)} OK, {len(errors)} ERROR ===\n")

print("OK:")
for r in ok:
    print(f"  {r['file']}: {r['char_count']} символів")

print(f"\nERROR:")
for r in errors:
    print(f"  {r['file']}: [{r['error_type']}] {r['error_message']}")

=== Результати: 15 OK, 10 ERROR ===

OK:
  boilerplate_heavy.html: 528 символів
  complex_merged_formulas.xlsx: 4996 символів
  empty_with_headers.xlsx: 25 символів
  financial_report_table.pdf: 704 символів
  huge_audit_log.txt: 1345999 символів
  huge_export.xlsx: 722153 символів
  huge_repetitive.html: 494324 символів
  latin1_mixed.html: 61 символів
  malformed_deeply_nested.html: 261 символів
  multilingual.html: 655 символів
  normal_report.xlsx: 310 символів
  password_protected.pdf: 0 символів
  scanned_no_text.pdf: 0 символів
  unicode_data.xlsx: 325 символів
  windows1251_no_charset.html: 96 символів

ERROR:
  actually_html.pdf: [type_mismatch] type mismatch
  actually_pdf.html: [type_mismatch] type mismatch
  binary_garbage.pdf: [type_mismatch] type mismatch
  broken_archive.docx: [type_mismatch] type mismatch
  corrupted.xlsx: [type_mismatch] type mismatch
  corrupted_truncated.pdf: [parse_error] Unable to get page count.
Syntax Error: Couldn't find trailer dictionary
Synta

---
## Завдання 5: Витягування таблиць з PDF

`unstructured` витягує текст з PDF, але **губить структуру таблиць** — рядки і колонки перемішуються в суцільний текст. Для таблиць потрібен спеціальний інструмент — `pdfplumber`.

**Файл для роботи:**
- `financial_report_table.pdf` — фінансовий звіт з двома таблицями (revenue by region, revenue by product)

**Що потрібно зробити:**
1. Витягти таблиці з PDF через `pdfplumber`
2. Перетворити кожну таблицю в список словників (перший рядок — ключі)
3. Порівняти результат з тим що дає `unstructured` — побачити різницю

In [14]:
import pdfplumber

def extract_tables_from_pdf(file_path: str) -> list[list[dict]]:
    """
    Витягує таблиці з PDF і повертає як список таблиць.
    Кожна таблиця — список словників (ключі = заголовки першого рядка).

    Приклад:
    [
        [  # таблиця 1
            {"Region": "North America", "Q1": "1,200,000", ...},
            {"Region": "Europe", "Q1": "800,000", ...},
        ],
        [  # таблиця 2
            {"Product": "Enterprise Platform", "Units Sold": "145", ...},
        ],
    ]
    """
    all_tables = []

    # TODO: відкрити PDF через pdfplumber.open(file_path)
    # TODO: для кожної сторінки — page.extract_tables()
    # TODO: для кожної raw таблиці:
    #   - перший рядок (raw_table[0]) — це заголовки (ключі словника)
    #   - решта рядків — дані
    #   - створити список словників з zip(headers, row) для кожного рядка
    # TODO: додати результат в all_tables

    settings_variants = [
        {
            "vertical_strategy": "lines",
            "horizontal_strategy": "lines",
            "snap_tolerance": 3,
            "join_tolerance": 3,
            "intersection_tolerance": 3,
        },
        {
            "vertical_strategy": "text",
            "horizontal_strategy": "text",
            "text_x_tolerance": 2,
            "text_y_tolerance": 2,
        },
    ]

    with pdfplumber.open(file_path) as pdf:
        for page in pdf.pages:
            raw_tables = []

            for settings in settings_variants:
                raw_tables = page.extract_tables(table_settings=settings)
                if raw_tables:
                    break

            for raw_table in raw_tables:
                if not raw_table or len(raw_table) < 2:
                    continue

                headers = raw_table[0]
                if not headers:
                    continue

                clean_headers = []
                for i, h in enumerate(headers):
                    h = "" if h is None else str(h).strip()
                    clean_headers.append(h if h else f"column_{i}")

                rows = []
                for row in raw_table[1:]:
                    if not row:
                        continue

                    row = list(row)

                    if len(row) < len(clean_headers):
                        row += [""] * (len(clean_headers) - len(row))
                    elif len(row) > len(clean_headers):
                        row = row[:len(clean_headers)]

                    row_dict = {
                        clean_headers[i]: ("" if row[i] is None else str(row[i]).strip())
                        for i in range(len(clean_headers))
                    }

                    if any(v != "" for v in row_dict.values()):
                        rows.append(row_dict)

                if rows:
                    all_tables.append(rows)

    return all_tables

In [15]:
pdf_path = "samples/enterprise_challenges/financial_report_table.pdf"

# --- pdfplumber: структуровані таблиці ---
tables = extract_tables_from_pdf(pdf_path)
print(f"pdfplumber знайшов {len(tables)} таблиць\n")

for i, table in enumerate(tables):
    print(f"=== Таблиця {i+1}: {len(table)} рядків ===")
    if table:
        print(f"  Колонки: {list(table[0].keys())}")
        for row in table[:3]:
            print(f"  {row}")
        if len(table) > 3:
            print(f"  ... ще {len(table) - 3} рядків")
    print()

# --- unstructured: для порівняння ---
print("=== unstructured (для порівняння) ===")
elements = partition(filename=pdf_path)
text = "\n".join(str(el) for el in elements)
print(text[:500])
print("\n^ Бачите різницю? unstructured втрачає структуру таблиці.")

pdfplumber знайшов 2 таблиць

=== Таблиця 1: 5 рядків ===
  Колонки: ['Region', 'Q1', 'Q2', 'Q3', 'Q4', 'Total']
  {'Region': 'North America', 'Q1': '1,200,000', 'Q2': '1,350,000', 'Q3': '1,100,000', 'Q4': '1,500,000', 'Total': '5,150,000'}
  {'Region': 'Europe', 'Q1': '800,000', 'Q2': '920,000', 'Q3': '870,000', 'Q4': '1,050,000', 'Total': '3,640,000'}
  {'Region': 'APAC', 'Q1': '450,000', 'Q2': '520,000', 'Q3': '610,000', 'Q4': '780,000', 'Total': '2,360,000'}
  ... ще 2 рядків

=== Таблиця 2: 4 рядків ===
  Колонки: ['Product', 'Units Sold', 'Revenue', 'Avg Price', 'Margin %']
  {'Product': 'Enterprise Platform', 'Units Sold': '145', 'Revenue': '4,350,000', 'Avg Price': '30,000', 'Margin %': '72%'}
  {'Product': 'SMB Suite', 'Units Sold': '1,230', 'Revenue': '3,690,000', 'Avg Price': '3,000', 'Margin %': '65%'}
  {'Product': 'API Access', 'Units Sold': '8,500', 'Revenue': '2,550,000', 'Avg Price': '300', 'Margin %': '88%'}
  ... ще 1 рядків

=== unstructured (для порівняння) ===


Q4 2025 Financial Report
Revenue breakdown by region and product line.
Table 1: Quarterly Revenue by Region (USD)
Region
Q1
Q2
Q3
North America
1,200,000
1,350,000
1,100,000
Europe
800,000
920,000
870,000
APAC
450,000
520,000
610,000
LATAM
200,000
230,000
250,000
Total
2,650,000
3,020,000
2,830,000
Table 2: Revenue by Product Line
Product
Units Sold
Revenue
Enterprise Platform
145
4,350,000
SMB Suite
1,230
3,690,000
API Access
8,500
2,550,000
Consulting
310
1,550,000
Note: All figures are unaudi

^ Бачите різницю? unstructured втрачає структуру таблиці.


---
## Завдання 6: Chunking великого документа

Для RAG та embeddings текст треба розбити на чанки. Але з великим файлом (1MB+) є нюанси:
- Як розмір чанка впливає на кількість чанків?
- Що відбувається з overlap?
- Скільки часу займає chunking?

**Файл для роботи:**
- `huge_audit_log.txt` — ~1.5 MB, 5000 записів audit log

**Що потрібно зробити:**
1. Реалізувати chunking через `RecursiveCharacterTextSplitter`
2. Порівняти різні `chunk_size` (256, 512, 1024, 2048) — кількість чанків, час
3. Порівняти різні `chunk_overlap` (0, 50, 200) — як overlap впливає на кількість чанків

In [16]:
huge_file = Path("samples/enterprise_challenges/huge_audit_log.txt")
text = huge_file.read_text()
size_mb = huge_file.stat().st_size / 1024 / 1024
print(f"Файл: {huge_file.name}")
print(f"Розмір: {size_mb:.2f} MB, {len(text):,} символів")

Файл: huge_audit_log.txt
Розмір: 1.29 MB, 1,350,998 символів


In [17]:
import time
from langchain_text_splitters import RecursiveCharacterTextSplitter


def chunk_text(text: str, chunk_size: int = 512, chunk_overlap: int = 50) -> list[str]:
    """
    Розбиває текст на чанки.
    Повертає список рядків (чанків).
    """
    # TODO: створити RecursiveCharacterTextSplitter з chunk_size та chunk_overlap
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )

    # TODO: розбити текст через splitter.split_text(text) і повернути результат
    chunks = splitter.split_text(text)

    return chunks

In [18]:
# Експеримент 1: різний chunk_size (overlap=50)
print(f"Текст: {len(text):,} символів\n")
print(f"{'chunk_size':>12} | {'чанків':>8} | {'сер. довж.':>10} | {'час (мс)':>10}")
print("-" * 52)

for size in [256, 512, 1024, 2048]:
    t0 = time.time()
    chunks = chunk_text(text, chunk_size=size, chunk_overlap=50)
    elapsed = (time.time() - t0) * 1000
    avg = sum(len(c) for c in chunks) / len(chunks) if chunks else 0
    print(f"{size:>12} | {len(chunks):>8} | {avg:>10.0f} | {elapsed:>10.1f}")

Текст: 1,350,998 символів

  chunk_size |   чанків | сер. довж. |   час (мс)
----------------------------------------------------
         256 |     9000 |        170 |      166.2
         512 |     4000 |        336 |        9.5
        1024 |     1667 |        808 |        8.0
        2048 |      715 |       1888 |        8.6


In [19]:
# Експеримент 2: різний chunk_overlap (chunk_size=512)
print(f"chunk_size=512, текст: {len(text):,} символів\n")
print(f"{'overlap':>12} | {'чанків':>8} | {'додатково':>12} | {'час (мс)':>10}")
print("-" * 52)

baseline = None
for overlap in [0, 50, 100, 200]:
    t0 = time.time()
    chunks = chunk_text(text, chunk_size=512, chunk_overlap=overlap)
    elapsed = (time.time() - t0) * 1000
    if baseline is None:
        baseline = len(chunks)
    extra = len(chunks) - baseline
    print(f"{overlap:>12} | {len(chunks):>8} | +{extra:>10} | {elapsed:>10.1f}")

print(f"\n(overlap збільшує кількість чанків, бо текст дублюється на стиках)")

chunk_size=512, текст: 1,350,998 символів

     overlap |   чанків |    додатково |   час (мс)
----------------------------------------------------
           0 |     4000 | +         0 |       13.3
          50 |     4000 | +         0 |        9.1
         100 |     4000 | +         0 |        9.0
         200 |     4000 | +         0 |        8.4

(overlap збільшує кількість чанків, бо текст дублюється на стиках)


In [20]:
# Подивимось на стик між чанками — як працює overlap
chunks = chunk_text(text, chunk_size=512, chunk_overlap=100)
print(f"Всього чанків: {len(chunks)}\n")

print("=== Чанк 0 (останні 150 символів) ===")
print(f"...{chunks[0][-150:]}")
print(f"\n=== Чанк 1 (перші 150 символів) ===")
print(f"{chunks[1][:150]}...")
print(f"\n^ Бачите overlap? Кінець чанка 0 повторюється на початку чанка 1.")
print("  Це потрібно щоб контекст не губився на стиках.")

Всього чанків: 4000

=== Чанк 0 (останні 150 символів) ===
...ear-over-year, driven by expansion into Southeast Asian markets. Operating margins improved from 22% to 27%, reflecting cost optimization initiatives.

=== Чанк 1 (перші 150 символів) ===
[Entry 00002] Customer acquisition costs decreased by 12% while retention rates improved to 94%. The enterprise segment showed particularly strong per...

^ Бачите overlap? Кінець чанка 0 повторюється на початку чанка 1.
  Це потрібно щоб контекст не губився на стиках.


---
## Готово!

Ви навчились:
1. **Визначати кодування** — charset_normalizer, BOM, legacy encodings
2. **Перевіряти тип файлу** — magic bytes vs розширення
3. **Чистити HTML** — видаляти шум, витягувати корисний текст
4. **Безпечно парсити** — обробляти corrupted/empty/broken файли без crash
5. **Витягувати таблиці з PDF** — pdfplumber vs unstructured, структуровані дані
6. **Chunking** — як chunk_size і overlap впливають на результат для великих документів

In [21]:
!python evaluate.py

Завантажую функції з homework.ipynb...

/usr/bin/python3
3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Файли для роботи:
  actually_html.pdf                                99 bytes
  actually_pdf.html                                66 bytes
  binary_garbage.pdf                            1,024 bytes
  boilerplate_heavy.html                        5,344 bytes
  broken_archive.docx                           4,400 bytes
  complex_merged_formulas.xlsx                  9,275 bytes
  corrupted.xlsx                                5,100 bytes
  corrupted_truncated.pdf                         335 bytes
  empty_file.docx                                   0 bytes
  empty_file.html                                   0 bytes
  empty_file.pdf                                    0 bytes
  empty_with_headers.xlsx                       4,862 bytes
  financial_report_table.pdf                    2,783 bytes
  huge_audit_log.txt                        1,350,998 bytes
  huge_export.xlsx                